In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import pandas as pd
file_path = "/content/drive/MyDrive/job/OFO DNA Collection Statistics Jan 1, 2025 Through January 31, 2026.csv"
df = pd.read_csv(file_path)


In [3]:
df.columns = [
    "collection_id", "dup_indicator", "port_name", "date", "disposition",
    "age", "citizenship", "custody_status", "transfer_agency",
    "transfer_location", "presented_ausa", "ausa_decision", "charge", "qualifying_reason"
]

In [4]:
df_persons = df[df["dup_indicator"].isna()].copy()

print(f"Total rows:        {len(df):,}")
print(f"Distinct persons:  {len(df_persons):,}")
print(f"Duplicate rows:    {len(df) - len(df_persons):,} (same person, multiple charges)")

Total rows:        171,048
Distinct persons:  79,363
Duplicate rows:    91,685 (same person, multiple charges)


In [5]:
def show_counts(series, title):
    print(f"\n{'='*60}")
    print(title)
    print(f"{'='*60}")
    counts = series.value_counts(dropna=False)
    counts.index = counts.index.fillna("(blank / not recorded)")
    pct = (counts / counts.sum() * 100).round(2)
    result = pd.DataFrame({"Count": counts, "% of Total": pct})
    print(result.to_string())
    print(f"\nTotal: {counts.sum():,}")

In [6]:
show_counts(df_persons["port_name"],             "COUNT BY FIELD OFFICE")


COUNT BY FIELD OFFICE
               Count  % of Total
port_name                       
LAREDO         19010       23.95
SAN DIEGO      11371       14.33
EL PASO         8837       11.13
HOUSTON         4959        6.25
TUCSON          4571        5.76
MIAMI           4082        5.14
DETROIT         3457        4.36
SEATTLE         3078        3.88
NEW YORK        2904        3.66
CHICAGO         2854        3.60
BUFFALO         2622        3.30
SAN FRANCISCO   2339        2.95
ATLANTA         2158        2.72
BOSTON          1666        2.10
LOS ANGELES     1421        1.79
SAN JUAN        1379        1.74
BALTIMORE       1228        1.55
TAMPA           1174        1.48
PORTLAND         206        0.26
PRECLEARANCE      32        0.04
NEW ORLEANS       15        0.02

Total: 79,363


In [7]:
show_counts(df_persons["age"],             "COUNT BY AGE AT EVENT CREATION (distinct persons)")


COUNT BY AGE AT EVENT CREATION (distinct persons)
                        Count  % of Total
age                                      
29.0                     2613        3.29
25.0                     2570        3.24
32.0                     2546        3.21
28.0                     2529        3.19
30.0                     2495        3.14
31.0                     2488        3.13
34.0                     2485        3.13
26.0                     2443        3.08
27.0                     2440        3.07
33.0                     2423        3.05
35.0                     2343        2.95
24.0                     2320        2.92
36.0                     2249        2.83
23.0                     2142        2.70
37.0                     2082        2.62
22.0                     1988        2.50
38.0                     1985        2.50
39.0                     1935        2.44
41.0                     1916        2.41
40.0                     1862        2.35
21.0                     

In [8]:
show_counts(df_persons["citizenship"],     "COUNT BY CITIZENSHIP COUNTRY (distinct persons)")


COUNT BY CITIZENSHIP COUNTRY (distinct persons)
                                      Count  % of Total
citizenship                                            
MEXICO                                26838       33.82
VENEZUELA                              8572       10.80
CANADA                                 6121        7.71
CUBA                                   4941        6.23
COLOMBIA                               3154        3.97
HONDURAS                               2451        3.09
CHINA                                  2048        2.58
GUATEMALA                              1605        2.02
HAITI                                  1490        1.88
SOUTH KOREA                            1450        1.83
INDIA                                  1416        1.78
SPAIN                                  1407        1.77
BRAZIL                                 1149        1.45
ECUADOR                                1110        1.40
DOMINICAN REPUBLIC                     1065        1.34

In [9]:
show_counts(df_persons["custody_status"],  "COUNT BY LATEST CUSTODY STATUS (distinct persons)")


COUNT BY LATEST CUSTODY STATUS (distinct persons)
                        Count  % of Total
custody_status                           
RETURNED FOREIGN        50417       63.53
RELEASED                21984       27.70
TURNED OVER TO           6928        8.73
NOT IN CUSTODY             19        0.02
(blank / not recorded)      9        0.01
CURRENTLY IN CUSTODY        3        0.00
TRANSPORTED                 3        0.00

Total: 79,363


In [10]:
show_counts(df_persons["transfer_agency"], "COUNT BY I216 CUSTODY TRANSFER TO (AGENCY) (distinct persons)")


COUNT BY I216 CUSTODY TRANSFER TO (AGENCY) (distinct persons)
                              Count  % of Total
transfer_agency                                
RETURN TO FOREIGN             50404       63.51
(blank / not recorded)        22028       27.76
ICE-ERO                        4803        6.05
FEDERAL/STATE/LOCAL FACILITY   1740        2.19
USBP                            273        0.34
ICE-HSI                         112        0.14
OFO                               3        0.00

Total: 79,363


In [11]:
show_counts(df_persons["transfer_location"], "COUNT BY I216 CUSTODY TRANSFER TO (LOCATION) (distinct persons)")


COUNT BY I216 CUSTODY TRANSFER TO (LOCATION) (distinct persons)
                                                                                                    Count  % of Total
transfer_location                                                                                                    
MEXICO                                                                                              22176       27.94
(blank / not recorded)                                                                              22028       27.76
CANADA                                                                                               7328        9.23
COLOMBIA                                                                                             2064        2.60
KOREA, REPUBLIC OF                                                                                   1450        1.83
CHINA (MAINLAND)                                                                                     1097    

In [12]:
show_counts(df_persons["presented_ausa"],  "PRESENTED TO AUSA (distinct persons)")


PRESENTED TO AUSA (distinct persons)
                        Count  % of Total
presented_ausa                           
(blank / not recorded)  76615       96.54
YES                      2033        2.56
NO                        715        0.90

Total: 79,363


In [13]:
show_counts(
    df_persons.loc[df_persons["presented_ausa"] == "YES", "ausa_decision"],
    "AUSA DECISION (distinct persons where Presented To AUSA = YES)"
)



AUSA DECISION (distinct persons where Presented To AUSA = YES)
               Count  % of Total
ausa_decision                   
ACCEPTED        1568       77.13
DECLINED         465       22.87

Total: 2,033


In [14]:
# Custody status by AUSA presentation status

# Q1: AUSA = Blank → custody status
show_counts(
    df_persons[df_persons["presented_ausa"].isna()]["custody_status"],
    "CUSTODY STATUS — Presented to AUSA = BLANK"
)

# Q2: Presented = NO → custody status
show_counts(
    df_persons[df_persons["presented_ausa"] == "NO"]["custody_status"],
    "CUSTODY STATUS — Presented to AUSA = NO"
)

# Q3: Presented = YES AND Decision = DECLINED → custody status
show_counts(
    df_persons[
        (df_persons["presented_ausa"] == "YES") &
        (df_persons["ausa_decision"] == "DECLINED")
    ]["custody_status"],
    "CUSTODY STATUS — Presented = YES, Decision = DECLINED"
)


CUSTODY STATUS — Presented to AUSA = BLANK
                        Count  % of Total
custody_status                           
RETURNED FOREIGN        49140       64.14
RELEASED                21961       28.66
TURNED OVER TO           5487        7.16
NOT IN CUSTODY             14        0.02
(blank / not recorded)      9        0.01
TRANSPORTED                 3        0.00
CURRENTLY IN CUSTODY        1        0.00

Total: 76,615

CUSTODY STATUS — Presented to AUSA = NO
                  Count  % of Total
custody_status                     
RETURNED FOREIGN    588       82.24
TURNED OVER TO      109       15.24
RELEASED             14        1.96
NOT IN CUSTODY        4        0.56

Total: 715

CUSTODY STATUS — Presented = YES, Decision = DECLINED
                  Count  % of Total
custody_status                     
RETURNED FOREIGN    403       86.67
TURNED OVER TO       58       12.47
RELEASED              3        0.65
NOT IN CUSTODY        1        0.22

Total: 465


In [15]:
# Q4: Combined group size and % of whole

mask_blank    = df_persons["presented_ausa"].isna()
mask_no       = df_persons["presented_ausa"] == "NO"
mask_declined = (df_persons["presented_ausa"] == "YES") & (df_persons["ausa_decision"] == "DECLINED")
mask_combined = mask_blank | mask_no | mask_declined

total = len(df_persons)

print(f"\n{'='*60}")
print("Q4: COMBINED GROUP COUNTS")
print(f"{'='*60}")
print(f"  AUSA = Blank:        {mask_blank.sum():>6,}  ({mask_blank.sum()/total*100:.2f}%)")
print(f"  Presented = NO:      {mask_no.sum():>6,}  ({mask_no.sum()/total*100:.2f}%)")
print(f"  YES + Declined:      {mask_declined.sum():>6,}  ({mask_declined.sum()/total*100:.2f}%)")
print(f"  ─────────────────────────────")
print(f"  Combined total:      {mask_combined.sum():>6,}  ({mask_combined.sum()/total*100:.2f}%)")
print(f"  Remaining (accepted): {(~mask_combined).sum():>5,}  ({(~mask_combined).sum()/total*100:.2f}%)")
print(f"  Total persons:       {total:>6,}")


Q4: COMBINED GROUP COUNTS
  AUSA = Blank:        76,615  (96.54%)
  Presented = NO:         715  (0.90%)
  YES + Declined:         465  (0.59%)
  ─────────────────────────────
  Combined total:      77,795  (98.02%)
  Remaining (accepted): 1,568  (1.98%)
  Total persons:       79,363


In [16]:
# Q5

# Qualifying reason — distinct persons in combined group
show_counts(
    df_persons[mask_combined]["qualifying_reason"],
    "QUALIFYING REASON — Combined group (blank + NO + declined)"
)

# Charges — need to pull from the full df (which has one row per charge)
# Re-apply the combined mask using the index of df_persons
combined_ids = df_persons[mask_combined].index

show_counts(
    df.loc[df.index.isin(combined_ids), "charge"],
    "CHARGES — Combined group (all charge rows, each counted separately)"
)


QUALIFYING REASON — Combined group (blank + NO + declined)
                         Count  % of Total
qualifying_reason                         
DETAINEE                 74245       95.44
ARRESTEE/FACING CHARGES   3550        4.56

Total: 77,795

CHARGES — Combined group (all charge rows, each counted separately)
                                                                                                             Count  % of Total
charge                                                                                                                        
8 USC 1182(A)-ALIEN INADMISSIBILITY UNDER SECTION 212(A)                                                     75214       96.68
8 USC 1227-DEPORTABLE ALIEN                                                                                   1144        1.47
8 USC 1229A-ALIEN REMOVAL UNDER SECTION 212/237                                                                718        0.92
8 USC 1282(B)-CONDITIONAL PERMITS TO LAND TEMPORA

In [17]:
show_counts(df["charge"], "COUNT BY CHARGE (all rows, each charge counted separately)")


COUNT BY CHARGE (all rows, each charge counted separately)
                                                                                                                                                                               Count  % of Total
charge                                                                                                                                                                                          
8 USC 1182(A)-ALIEN INADMISSIBILITY UNDER SECTION 212(A)                                                                                                                       76735       44.86
SEC212(A)(7)(A)(I)(I)-IMMIGRANT W/OUT DOCS                                                                                                                                     65959       38.56
SEC212(A)(7)(B)(I)(II)-NONIMMIGRANT W/OUT DOCS                                                                                                                          

In [18]:
show_counts(df_persons["qualifying_reason"], "COUNT BY QUALIFYING REASON FOR DNA COLLECTION (distinct persons)")



COUNT BY QUALIFYING REASON FOR DNA COLLECTION (distinct persons)
                         Count  % of Total
qualifying_reason                         
DETAINEE                 74932       94.42
ARRESTEE/FACING CHARGES   4431        5.58

Total: 79,363


In [19]:
# Parse date column and create 2025 calendar-year filter
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# 2025 calendar year: Jan 1, 2025 – Dec 31, 2025
mask_2025 = (df['date'] >= '2025-01-01') & (df['date'] <= '2025-12-31')
df_2025 = df[mask_2025].copy()
df_persons_2025 = df_2025[df_2025['dup_indicator'].isna()].copy()

print()
print(f"--- 2025 CALENDAR YEAR ONLY (Jan 1 – Dec 31, 2025) ---")
print(f"Total rows:        {len(df_2025):,}")
print(f"Distinct persons:  {len(df_persons_2025):,}")
print(f"Duplicate rows:    {len(df_2025) - len(df_persons_2025):,}")



--- 2025 CALENDAR YEAR ONLY (Jan 1 – Dec 31, 2025) ---
Total rows:        161,651
Distinct persons:  75,215
Duplicate rows:    86,436


In [20]:
show_counts(df_persons_2025["age"],              "COUNT BY AGE (distinct persons — 2025 calendar year)")



COUNT BY AGE (distinct persons — 2025 calendar year)
                        Count  % of Total
age                                      
29.0                     2493        3.31
25.0                     2454        3.26
32.0                     2402        3.19
28.0                     2402        3.19
30.0                     2392        3.18
31.0                     2381        3.17
34.0                     2366        3.15
26.0                     2315        3.08
27.0                     2314        3.08
33.0                     2297        3.05
35.0                     2232        2.97
24.0                     2216        2.95
36.0                     2141        2.85
23.0                     2027        2.69
37.0                     1953        2.60
22.0                     1890        2.51
38.0                     1871        2.49
39.0                     1846        2.45
41.0                     1809        2.41
40.0                     1760        2.34
21.0                  

In [21]:
show_counts(df_persons_2025["citizenship"],      "COUNT BY CITIZENSHIP COUNTRY (distinct persons — 2025 calendar year)")



COUNT BY CITIZENSHIP COUNTRY (distinct persons — 2025 calendar year)
                                      Count  % of Total
citizenship                                            
MEXICO                                24967       33.19
VENEZUELA                              8542       11.36
CANADA                                 5720        7.60
CUBA                                   4909        6.53
COLOMBIA                               3025        4.02
HONDURAS                               2403        3.19
CHINA                                  1880        2.50
GUATEMALA                              1555        2.07
HAITI                                  1484        1.97
SOUTH KOREA                            1310        1.74
SPAIN                                  1306        1.74
INDIA                                  1289        1.71
BRAZIL                                 1088        1.45
ECUADOR                                1074        1.43
DOMINICAN REPUBLIC                

In [22]:
show_counts(df_persons_2025["custody_status"],   "COUNT BY LATEST CUSTODY STATUS (distinct persons — 2025 calendar year)")



COUNT BY LATEST CUSTODY STATUS (distinct persons — 2025 calendar year)
                        Count  % of Total
custody_status                           
RETURNED FOREIGN        46682       62.06
RELEASED                21958       29.19
TURNED OVER TO           6545        8.70
NOT IN CUSTODY             19        0.03
(blank / not recorded)      9        0.01
CURRENTLY IN CUSTODY        2        0.00

Total: 75,215


In [23]:
show_counts(df_persons_2025["transfer_agency"],  "COUNT BY I216 CUSTODY TRANSFER TO (AGENCY) (distinct persons — 2025 calendar year)")



COUNT BY I216 CUSTODY TRANSFER TO (AGENCY) (distinct persons — 2025 calendar year)
                              Count  % of Total
transfer_agency                                
RETURN TO FOREIGN             46671       62.05
(blank / not recorded)        21999       29.25
ICE-ERO                        4595        6.11
FEDERAL/STATE/LOCAL FACILITY   1586        2.11
USBP                            263        0.35
ICE-HSI                         101        0.13

Total: 75,215


In [24]:
show_counts(df_persons_2025["transfer_location"],"COUNT BY I216 CUSTODY TRANSFER TO (LOCATION) (distinct persons — 2025 calendar year)")



COUNT BY I216 CUSTODY TRANSFER TO (LOCATION) (distinct persons — 2025 calendar year)
                                                                                                    Count  % of Total
transfer_location                                                                                                    
(blank / not recorded)                                                                              21999       29.25
MEXICO                                                                                              20458       27.20
CANADA                                                                                               6850        9.11
COLOMBIA                                                                                             1926        2.56
KOREA, REPUBLIC OF                                                                                   1320        1.75
BRAZIL                                                                                  

In [25]:
show_counts(df_persons_2025["presented_ausa"],   "PRESENTED TO AUSA (distinct persons — 2025 calendar year)")



PRESENTED TO AUSA (distinct persons — 2025 calendar year)
                        Count  % of Total
presented_ausa                           
(blank / not recorded)  72692       96.65
YES                      1872        2.49
NO                        651        0.87

Total: 75,215


In [26]:
show_counts(
    df_persons_2025.loc[df_persons_2025["presented_ausa"] == "YES", "ausa_decision"],
    "AUSA DECISION (distinct persons where Presented To AUSA = YES — 2025 calendar year)"
)



AUSA DECISION (distinct persons where Presented To AUSA = YES — 2025 calendar year)
               Count  % of Total
ausa_decision                   
ACCEPTED        1445       77.19
DECLINED         427       22.81

Total: 1,872


In [27]:
show_counts(df_2025["charge"],                   "COUNT BY CHARGE (all rows, each charge counted separately — 2025 calendar year)")



COUNT BY CHARGE (all rows, each charge counted separately — 2025 calendar year)
                                                                                                                                                                               Count  % of Total
charge                                                                                                                                                                                          
8 USC 1182(A)-ALIEN INADMISSIBILITY UNDER SECTION 212(A)                                                                                                                       72736       45.00
SEC212(A)(7)(A)(I)(I)-IMMIGRANT W/OUT DOCS                                                                                                                                     62771       38.83
SEC212(A)(7)(B)(I)(II)-NONIMMIGRANT W/OUT DOCS                                                                                                     

In [28]:
show_counts(df_persons_2025["qualifying_reason"],"COUNT BY QUALIFYING REASON (distinct persons — 2025 calendar year)")


COUNT BY QUALIFYING REASON (distinct persons — 2025 calendar year)
                         Count  % of Total
qualifying_reason                         
DETAINEE                 71052       94.47
ARRESTEE/FACING CHARGES   4163        5.53

Total: 75,215


In [34]:
# ══════════════════════════════════════════════════════════════
# TOP CHARGES + FIELD OFFICES BY COUNTRY
# ══════════════════════════════════════════════════════════════

target_countries = ["CANADA", "SOUTH KOREA", "CHINA"]

for country in target_countries:
    # Get distinct persons from that country
    country_persons = df_persons[df_persons["citizenship"] == country]

    # Use collection_id instead of index
    country_ids = country_persons["collection_id"]

    # Pull all charge rows for those persons
    country_charges = df[
        df["collection_id"].isin(country_ids)
    ]["charge"]

    # Pull field offices (person-level is cleaner here)
    country_ports = country_persons["port_name"]

    print(f"\n{'='*60}")
    print(f"{country} — OVERVIEW")
    print(f"{'='*60}")
    print(f"  Distinct persons : {len(country_persons):,}")
    print(f"  Charge rows      : {len(country_charges):,}")

    # Top charges
    show_counts(
        country_charges,
        f"TOP CHARGES — {country}"
    )

    # Field offices
    show_counts(
        country_ports,
        f"CBP FIELD OFFICES — {country}"
    )


CANADA — OVERVIEW
  Distinct persons : 6,121
  Charge rows      : 171,048

TOP CHARGES — CANADA
                                                                                                                                                                               Count  % of Total
charge                                                                                                                                                                                          
8 USC 1182(A)-ALIEN INADMISSIBILITY UNDER SECTION 212(A)                                                                                                                       76735       44.86
SEC212(A)(7)(A)(I)(I)-IMMIGRANT W/OUT DOCS                                                                                                                                     65959       38.56
SEC212(A)(7)(B)(I)(II)-NONIMMIGRANT W/OUT DOCS                                                                                     

In [30]:
# ══════════════════════════════════════════════════════════════
# MINORS UNDER 14: TOP CHARGES
# ══════════════════════════════════════════════════════════════

minors = df_persons[df_persons["age"] < 14]
minor_ids = minors.index
minor_charges = df[df.index.isin(minor_ids)]["charge"]

print(f"\nDistinct minors under 14: {len(minors):,}")
print(f"Total charge rows:        {len(minor_charges):,}")

show_counts(
    minor_charges,
    "TOP CHARGES — Minors under 14"
)


Distinct minors under 14: 492
Total charge rows:        492

TOP CHARGES — Minors under 14
                                                          Count  % of Total
charge                                                                     
8 USC 1182(A)-ALIEN INADMISSIBILITY UNDER SECTION 212(A)    465       94.51
8 USC 1227-DEPORTABLE ALIEN                                  14        2.85
8 USC 1229A-ALIEN REMOVAL UNDER SECTION 212/237              12        2.44
SEC212(A)(7)(A)(I)(I)-IMMIGRANT W/OUT DOCS                    1        0.20

Total: 492


In [31]:
# ══════════════════════════════════════════════════════════════
# MINORS UNDER 14: QUALIFYING REASON (detainee vs. arrestee)
# ══════════════════════════════════════════════════════════════

show_counts(
    minors["qualifying_reason"],
    "QUALIFYING REASON — Minors under 14 (distinct persons)"
)


QUALIFYING REASON — Minors under 14 (distinct persons)
                         Count  % of Total
qualifying_reason                         
DETAINEE                   485       98.58
ARRESTEE/FACING CHARGES      7        1.42

Total: 492


In [32]:
# ══════════════════════════════════════════════════════════════
# SECTION 212(F) INVASION PROCLAMATION — FULL PROFILE
# ══════════════════════════════════════════════════════════════

INVASION_CHARGE = "SEC212(F) PROCLAMATION: GUARANTEEING THE STATES PROTECTION AGAINST INVASION"

# Get all charge rows matching this charge
invasion_rows = df[df["charge"] == INVASION_CHARGE]

# Get unique person IDs from charge rows
invasion_ids = invasion_rows["collection_id"].unique()

# Filter distinct persons using collection_id
invasion_persons = df_persons[
    df_persons["collection_id"].isin(invasion_ids)
]
print(f"\n{'='*60}")
print("SEC 212(F) INVASION PROCLAMATION — PROFILE OVERVIEW")
print(f"{'='*60}")
print(f"  Total charge rows with this charge : {len(invasion_rows):,}")
print(f"  Distinct persons with this charge  : {len(invasion_persons):,}")

show_counts(
    invasion_persons["age"],
    "AGE — 212(f) Invasion persons"
)

show_counts(
    invasion_persons["citizenship"],
    "CITIZENSHIP — 212(f) Invasion persons"
)

show_counts(
    invasion_persons["qualifying_reason"],
    "QUALIFYING REASON — 212(f) Invasion persons"
)

show_counts(
    invasion_persons["custody_status"],
    "CUSTODY STATUS — 212(f) Invasion persons"
)

show_counts(
    invasion_persons["port_name"],
    "CBP FIELD OFFICE — 212(f) Invasion persons"
)

show_counts(
    invasion_persons["transfer_location"],
    "TRANSFER LOCATION — 212(f) Invasion persons"
)


SEC 212(F) INVASION PROCLAMATION — PROFILE OVERVIEW
  Total charge rows with this charge : 2,820
  Distinct persons with this charge  : 79,363

AGE — 212(f) Invasion persons
                        Count  % of Total
age                                      
29.0                     2613        3.29
25.0                     2570        3.24
32.0                     2546        3.21
28.0                     2529        3.19
30.0                     2495        3.14
31.0                     2488        3.13
34.0                     2485        3.13
26.0                     2443        3.08
27.0                     2440        3.07
33.0                     2423        3.05
35.0                     2343        2.95
24.0                     2320        2.92
36.0                     2249        2.83
23.0                     2142        2.70
37.0                     2082        2.62
22.0                     1988        2.50
38.0                     1985        2.50
39.0                     19

In [33]:
# ══════════════════════════════════════════════════════════════
# SIDE-BY-SIDE SUMMARY: all age groups at a glance
# ══════════════════════════════════════════════════════════════

groups = {
    "Under 14"  : df_persons[df_persons["age"] <  14],
    "Under 18"  : df_persons[df_persons["age"] <  18],
    "65+"       : df_persons[df_persons["age"] >= 65],
    "70+"       : df_persons[df_persons["age"] >= 70],
    "80+"       : df_persons[df_persons["age"] >= 80],
}

print(f"\n{'='*72}")
print("AGE GROUP SUMMARY")
print(f"{'='*72}")
print(f"  {'Group':<12} {'Count':>8}  {'% of all':>10}  {'Released':>10}  {'Ret. Foreign':>13}  {'Turned Over':>12}")
print(f"  {'-'*68}")

for label, grp in groups.items():
    n = len(grp)
    pct = n / total * 100
    n_rel = (grp["custody_status"].isin(["RELEASED", "NOT IN CUSTODY"])).sum()
    n_ret = (grp["custody_status"] == "RETURNED FOREIGN").sum()
    n_to  = (grp["custody_status"] == "TURNED OVER TO").sum()
    rel_pct = n_rel / n * 100 if n else 0
    ret_pct = n_ret / n * 100 if n else 0
    to_pct  = n_to  / n * 100 if n else 0
    print(f"  {label:<12} {n:>8,}  {pct:>9.2f}%  {n_rel:>5,} {rel_pct:>4.1f}%  {n_ret:>6,} {ret_pct:>4.1f}%  {n_to:>5,} {to_pct:>4.1f}%")


AGE GROUP SUMMARY
  Group           Count    % of all    Released   Ret. Foreign   Turned Over
  --------------------------------------------------------------------
  Under 14          492       0.62%      3  0.6%     407 82.7%     82 16.7%
  Under 18        2,822       3.56%  1,153 40.9%   1,136 40.3%    533 18.9%
  65+             2,019       2.54%    271 13.4%   1,557 77.1%    190  9.4%
  70+               878       1.11%    133 15.1%     668 76.1%     77  8.8%
  80+               100       0.13%     21 21.0%      68 68.0%     11 11.0%
